# Axioms of Probability

Companion notebook for [Axioms of Probability](../../notes/probability/axioms_of_probability.md).

We check Kolmogorov's axioms for a probability assignment, verify consequences (complement and addition rules, monotonicity, inclusion-exclusion) by exact enumeration, and illustrate them by simulation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from fractions import Fraction

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (7, 3.5)

## 1. Checking a probability assignment

An assignment on a finite sample space is valid when every probability is $\ge0$ and they sum to 1 (axioms I and II); axiom III then lets us obtain the probability of any event by adding outcome probabilities.

In [2]:
def is_valid(p, tol=1e-12):
    p = np.asarray(p, dtype=float)
    return bool((p >= 0).all() and abs(p.sum() - 1) < tol)

for label, p in [("(0.5, 0.3, 0.3)", [0.5, 0.3, 0.3]),
                 ("(0.4, 0.6, 0.0)", [0.4, 0.6, 0.0]),
                 ("(0.7, -0.1, 0.4)", [0.7, -0.1, 0.4])]:
    print(f"{label:18s} valid = {is_valid(p)}")

(0.5, 0.3, 0.3)    valid = False
(0.4, 0.6, 0.0)    valid = True
(0.7, -0.1, 0.4)   valid = False


## 2. Events on a card deck

We build the 52-card deck as pairs (rank, suit) and verify the complement rule and the general addition rule $P(A\cup B)=P(A)+P(B)-P(A\cap B)$ for "heart" and "queen".

In [3]:
ranks = ["2","3","4","5","6","7","8","9","10","J","Q","K","A"]
suits = ["clubs", "diamonds", "hearts", "spades"]
deck = list(product(ranks, suits))
P = lambda ev: Fraction(sum(1 for c in deck if ev(c)), len(deck))

heart = lambda c: c[1] == "hearts"
queen = lambda c: c[0] == "Q"
print("P(heart)         =", P(heart))
print("P(not heart)     =", P(lambda c: not heart(c)), "= 1 - P(heart):", 1 - P(heart))
lhs = P(lambda c: heart(c) or queen(c))
rhs = P(heart) + P(queen) - P(lambda c: heart(c) and queen(c))
print("P(heart or queen) =", lhs, "=", rhs, "->", lhs == rhs)

red = lambda c: c[1] in ("diamonds", "hearts")
ace = lambda c: c[0] == "A"
print("P(ace or red) =", P(lambda c: ace(c) or red(c)))

P(heart)         = 1/4
P(not heart)     = 3/4 = 1 - P(heart): 3/4
P(heart or queen) = 4/13 = 4/13 -> True
P(ace or red) = 7/13


## 3. Mutually exclusive versus overlapping events

For mutually exclusive events probabilities simply add. For overlapping events we must subtract the intersection, otherwise outcomes are counted twice. On a die: "even or 3" are disjoint, "even or 4" overlap.

In [4]:
die = range(1, 7)
Pd = lambda ev: Fraction(sum(1 for x in die if ev(x)), 6)
even = lambda x: x % 2 == 0
print("P(even or 3) =", Pd(lambda x: even(x) or x == 3), "= P(even)+P(3) =", Pd(even) + Pd(lambda x: x == 3))
print("P(even or 4) =", Pd(lambda x: even(x) or x == 4), "!= P(even)+P(4) =", Pd(even) + Pd(lambda x: x == 4))

P(even or 3) = 2/3 = P(even)+P(3) = 2/3
P(even or 4) = 1/2 != P(even)+P(4) = 2/3


## 4. Inclusion-exclusion for three events

$$P(A\cup B\cup C)=\sum P(A_i)-\sum P(A_i\cap A_j)+P(A\cap B\cap C).$$

With $P(A)=0.5,P(B)=0.4,P(C)=0.3$, pairwise intersections $0.2,0.1,0.15$ and triple $0.05$ the union is 0.8. We verify the formula on a random finite probability space.

In [5]:
print("example:", 0.5 + 0.4 + 0.3 - 0.2 - 0.1 - 0.15 + 0.05)

# Random check: 8 outcomes with random probabilities, three random events
p = rng.dirichlet(np.ones(8))
A, B, Cc = (rng.random(8) < 0.3 for _ in range(3))
pr = lambda m: p[m].sum()
union = pr(A | B | Cc)
formula = pr(A) + pr(B) + pr(Cc) - pr(A & B) - pr(A & Cc) - pr(B & Cc) + pr(A & B & Cc)
print("direct union:", round(union, 6), " inclusion-exclusion:", round(formula, 6))

example: 0.8
direct union: 0.610679  inclusion-exclusion: 0.610679


## 5. Bounds on the intersection

Since $P(A\cup B)\le1$ we have $P(A\cap B)\ge P(A)+P(B)-1$ (Bonferroni), and $P(A\cap B)\le\min(P(A),P(B))$. For $P(A)=0.6$, $P(B)=0.7$ the intersection lies in $[0.3,0.6]$. We sample random joint distributions of $(A,B)$ to see that the bounds hold and are attained.

In [6]:
pA, pB = 0.6, 0.7
inter = []
for _ in range(20000):
    # random P(A and B) is feasible iff all four cell probabilities are >= 0
    x = rng.uniform(0, min(pA, pB))
    if pA - x >= 0 and pB - x >= 0 and 1 - pA - pB + x >= 0:
        inter.append(x)
inter = np.array(inter)
print("feasible range found: [%.3f, %.3f]" % (inter.min(), inter.max()))
print("theory: [%.3f, %.3f]" % (max(0, pA + pB - 1), min(pA, pB)))

feasible range found: [0.300, 0.600]
theory: [0.300, 0.600]
